In [0]:
# Databricks notebook source
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

print("=" * 60)
print("FASE 2: ANÁLISE EXPLORATÓRIA (EDA)")
print("=" * 60)

# Parâmetros do projeto
VOLUME_PATH = "/Volumes/workspace/default/nyc_taxi"
DELTA_PATH = f"{VOLUME_PATH}/raw/delta"

FILES = ["2015-01", "2016-01", "2016-02", "2016-03"]

# Carregar e combinar os 4 arquivos Delta
dfs = [spark.read.format("delta").load(f"{DELTA_PATH}/taxi_{periodo}") 
       for periodo in FILES]

df = dfs[0]
for d in dfs[1:]:
    df = df.unionByName(d)

df.createOrReplaceTempView("taxi_data")

total_linhas = df.count()
print(f"\n📊 Total combinado: {total_linhas:,} linhas")
print(f"📋 Colunas: {len(df.columns)}")

In [0]:
# TAREFA 2.2: Estatísticas Descritivas
print("\n" + "="*60)
print("TAREFA 2.2: Estatísticas Descritivas")
print("="*60)

display(df.select(
    "trip_distance", "fare_amount", "tip_amount", 
    "total_amount", "passenger_count"
).describe())

# Correlações principais
print("\n📊 Correlações:")
correlacoes = spark.sql("""
SELECT 
    ROUND(CORR(trip_distance, fare_amount), 3) as dist_fare_corr,
    ROUND(CORR(fare_amount, tip_amount), 3) as fare_tip_corr,
    ROUND(CORR(trip_distance, tip_amount), 3) as dist_tip_corr
FROM taxi_data
""")
display(correlacoes)

In [0]:
# TAREFA 2.3: Análise de Valores Nulos
print("\n" + "="*60)
print("TAREFA 2.3: Valores Nulos")
print("="*60)

nulos = spark.sql("""
SELECT 
    COUNT(*) as total_linhas,
    COUNT(trip_distance) as trip_distance_ok,
    COUNT(fare_amount) as fare_amount_ok,
    COUNT(tip_amount) as tip_amount_ok,
    COUNT(passenger_count) as passenger_count_ok,
    COUNT(dropoff_latitude) as dropoff_lat_ok,
    COUNT(dropoff_longitude) as dropoff_lng_ok,
    COUNT(pickup_latitude) as pickup_lat_ok,
    COUNT(pickup_longitude) as pickup_lng_ok
FROM taxi_data
""")
display(nulos)

print("\n💡 Comparando cada coluna com o total de linhas, identificamos % de nulos")

In [0]:
# TAREFA 2.4: Análise de Duplicatas
print("\n" + "="*60)
print("TAREFA 2.4: Duplicatas")
print("="*60)

duplicatas = spark.sql("""
SELECT COUNT(*) as total_duplicatas
FROM (
    SELECT tpep_pickup_datetime, tpep_dropoff_datetime, 
           pickup_latitude, pickup_longitude, fare_amount,
           COUNT(*) as qtd
    FROM taxi_data
    GROUP BY tpep_pickup_datetime, tpep_dropoff_datetime,
             pickup_latitude, pickup_longitude, fare_amount
    HAVING COUNT(*) > 1
)
""")

display(duplicatas)

In [0]:
# TAREFA 2.5: Análise Temporal
print("\n" + "="*60)
print("TAREFA 2.5: Padrões Temporais")
print("="*60)

df_temporal = df.withColumn(
    "pickup_hour", hour(col("tpep_pickup_datetime"))
).withColumn(
    "pickup_day_of_week", dayofweek(col("tpep_pickup_datetime"))
).withColumn(
    "pickup_year", year(col("tpep_pickup_datetime"))
)

df_temporal.createOrReplaceTempView("taxi_temporal")

# Viagens por hora do dia
print("\n📊 Viagens por hora do dia:")
por_hora = spark.sql("""
SELECT 
    pickup_hour,
    COUNT(*) as total_viagens,
    ROUND(AVG(fare_amount), 2) as tarifa_media,
    ROUND(AVG(tip_amount), 2) as gorjeta_media
FROM taxi_temporal
GROUP BY pickup_hour
ORDER BY pickup_hour
""")
display(por_hora)

In [0]:
# Continuação TAREFA 2.5: Dia da Semana
print("\n📊 Viagens por dia da semana:")

por_dia = spark.sql("""
SELECT 
    pickup_day_of_week,
    CASE 
        WHEN pickup_day_of_week = 1 THEN 'Domingo'
        WHEN pickup_day_of_week = 2 THEN 'Segunda'
        WHEN pickup_day_of_week = 3 THEN 'Terça'
        WHEN pickup_day_of_week = 4 THEN 'Quarta'
        WHEN pickup_day_of_week = 5 THEN 'Quinta'
        WHEN pickup_day_of_week = 6 THEN 'Sexta'
        ELSE 'Sábado'
    END as dia_semana,
    COUNT(*) as total_viagens,
    ROUND(AVG(fare_amount), 2) as tarifa_media
FROM taxi_temporal
GROUP BY pickup_day_of_week
ORDER BY pickup_day_of_week
""")
display(por_dia)

In [0]:
# Análise extra: 2015 vs 2016 (Janeiro)
print("\n" + "="*60)
print("Comparação Janeiro/2015 vs Janeiro/2016")
print("="*60)

comparacao_anos = spark.sql("""
SELECT 
    pickup_year,
    COUNT(*) as total_viagens,
    ROUND(AVG(fare_amount), 2) as tarifa_media,
    ROUND(AVG(tip_amount), 2) as gorjeta_media,
    ROUND(AVG(trip_distance), 2) as distancia_media
FROM taxi_temporal
WHERE MONTH(tpep_pickup_datetime) = 1
GROUP BY pickup_year
ORDER BY pickup_year
""")
display(comparacao_anos)

In [0]:
# TAREFA 2.6: Análise Geográfica
print("\n" + "="*60)
print("TAREFA 2.6: Padrões Geográficos")
print("="*60)

top_zonas = spark.sql("""
SELECT 
    ROUND(pickup_latitude, 2) as pickup_lat,
    ROUND(pickup_longitude, 2) as pickup_lng,
    COUNT(*) as total_viagens,
    ROUND(AVG(fare_amount), 2) as tarifa_media
FROM taxi_data
WHERE pickup_latitude IS NOT NULL 
  AND pickup_longitude IS NOT NULL
GROUP BY ROUND(pickup_latitude, 2), ROUND(pickup_longitude, 2)
ORDER BY total_viagens DESC
LIMIT 20
""")
display(top_zonas)

In [0]:
# TAREFA 2.7: Relação Distância-Tarifa-Gorjeta
print("\n" + "="*60)
print("TAREFA 2.7: Tarifa e Gorjeta por Distância")
print("="*60)

tarifa_distancia = spark.sql("""
SELECT 
    ROUND(trip_distance, 0) as distancia_milhas,
    COUNT(*) as total_viagens,
    ROUND(AVG(fare_amount), 2) as tarifa_media,
    ROUND(AVG(tip_amount), 2) as gorjeta_media,
    ROUND(AVG(tip_amount) / AVG(fare_amount) * 100, 1) as gorjeta_pct
FROM taxi_data
WHERE trip_distance BETWEEN 0.1 AND 30
GROUP BY ROUND(trip_distance, 0)
ORDER BY distancia_milhas
""")
display(tarifa_distancia)

In [0]:
# TAREFA 2.8: Análise de Pagamento
print("\n" + "="*60)
print("TAREFA 2.8: Tipo de Pagamento")
print("="*60)

pagamento = spark.sql("""
SELECT 
    payment_type,
    COUNT(*) as total_viagens,
    ROUND(AVG(fare_amount), 2) as tarifa_media,
    ROUND(AVG(tip_amount), 2) as gorjeta_media,
    ROUND(AVG(tip_amount) / AVG(fare_amount) * 100, 1) as gorjeta_pct
FROM taxi_data
GROUP BY payment_type
ORDER BY total_viagens DESC
""")
display(pagamento)

In [0]:
# TAREFA 2.9: Visualizações
print("\n" + "="*60)
print("TAREFA 2.9: Distribuição de Tarifas")
print("="*60)

distribuicao_tarifa = spark.sql("""
SELECT 
    ROUND(fare_amount, 0) as faixa_tarifa,
    COUNT(*) as total
FROM taxi_data
WHERE fare_amount BETWEEN 0 AND 100
GROUP BY ROUND(fare_amount, 0)
ORDER BY faixa_tarifa
""")

display(distribuicao_tarifa)
# 💡 Clique no ícone de gráfico abaixo da tabela e escolha "Bar Chart"
# eixo X: faixa_tarifa, eixo Y: total

Databricks visualization. Run in Databricks to view.

In [0]:
# Visualização: padrão horário
display(por_hora)
# 💡 Clique no ícone de gráfico e escolha "Line Chart"
# eixo X: pickup_hour, eixo Y: total_viagens

Databricks visualization. Run in Databricks to view.

In [0]:
# Investigação: Outliers extremos e diferença 2015 vs 2016
print("=" * 60)
print("INVESTIGAÇÃO: Outliers e Diferença Entre Anos")
print("=" * 60)

# 1. Percentis de trip_distance (visão sem distorção de outliers)
print("\n📊 Percentis de trip_distance:")
percentis = df.approxQuantile("trip_distance", [0.5, 0.9, 0.95, 0.99, 0.999], 0.01)
print(f"   Mediana (p50): {percentis[0]:.2f}")
print(f"   p90:  {percentis[1]:.2f}")
print(f"   p95:  {percentis[2]:.2f}")
print(f"   p99:  {percentis[3]:.2f}")
print(f"   p99.9: {percentis[4]:.2f}")

# 2. Quantos registros são outliers extremos?
outliers_distancia = spark.sql("""
SELECT COUNT(*) as qtd_outliers
FROM taxi_data
WHERE trip_distance > 200
""").collect()[0]["qtd_outliers"]
print(f"\n🚩 Viagens com distância > 200 milhas (~322km): {outliers_distancia:,}")

# 3. Comparar percentis de distância por ano (não só a média)
print("\n📊 Percentis de trip_distance por ano (em milhas):")
for ano_periodo in ["2015-01", "2016-01"]:
    df_ano = spark.read.format("delta").load(f"{DELTA_PATH}/taxi_{ano_periodo}")
    p = df_ano.approxQuantile("trip_distance", [0.5, 0.95, 0.99], 0.01)
    print(f"   {ano_periodo}: mediana={p[0]:.2f} mi, p95={p[1]:.2f} mi, p99={p[2]:.2f} mi")

# 4. Correlação sem outliers extremos (filtrando valores plausíveis)
print("\n📊 Correlação distância-tarifa (SEM outliers extremos):")
corr_limpa = spark.sql("""
SELECT ROUND(CORR(trip_distance, fare_amount), 3) as corr_limpa
FROM taxi_data
WHERE trip_distance BETWEEN 0.1 AND 100
  AND fare_amount BETWEEN 2.5 AND 500
""")
display(corr_limpa)

In [0]:
# Recalcular percentis SEM os outliers extremos primeiro
print("=" * 60)
print("PERCENTIS CORRIGIDOS (excluindo outliers >200km)")
print("=" * 60)

df_sem_outliers = df.filter(col("trip_distance") <= 200)

percentis_corrigidos = df_sem_outliers.approxQuantile(
    "trip_distance", [0.5, 0.9, 0.95, 0.99, 0.999], 0.001
)

print(f"\n📊 Percentis de trip_distance (dados limpos, em milhas):")
print(f"   Mediana (p50): {percentis_corrigidos[0]:.2f} milhas")
print(f"   p90:  {percentis_corrigidos[1]:.2f} milhas")
print(f"   p95:  {percentis_corrigidos[2]:.2f} milhas")
print(f"   p99:  {percentis_corrigidos[3]:.2f} milhas")
print(f"   p99.9: {percentis_corrigidos[4]:.2f} milhas")

# Mesma análise para fare_amount (também tinha outliers extremos)
print(f"\n📊 Percentis de fare_amount (removendo tarifas > $500):")
df_fare_limpo = df.filter((col("fare_amount") > 0) & (col("fare_amount") <= 500))
percentis_fare = df_fare_limpo.approxQuantile(
    "fare_amount", [0.5, 0.9, 0.95, 0.99, 0.999], 0.001
)
print(f"   Mediana: ${percentis_fare[0]:.2f}")
print(f"   p90:  ${percentis_fare[1]:.2f}")
print(f"   p95:  ${percentis_fare[2]:.2f}")
print(f"   p99:  ${percentis_fare[3]:.2f}")
print(f"   p99.9: ${percentis_fare[4]:.2f}")

# Quantos registros negativos existem em tip/total?
print(f"\n📊 Registros com valores negativos:")
negativos = spark.sql("""
SELECT 
    SUM(CASE WHEN tip_amount < 0 THEN 1 ELSE 0 END) as tip_negativo,
    SUM(CASE WHEN total_amount < 0 THEN 1 ELSE 0 END) as total_negativo,
    SUM(CASE WHEN fare_amount > 500 THEN 1 ELSE 0 END) as fare_extrema
FROM taxi_data
""")
display(negativos)

In [0]:
# Análise extra: Viagens por RatecodeID (identifica aeroportos)
print("\n" + "="*60)
print("ANÁLISE EXTRA: Viagens por RatecodeID")
print("="*60)

ratecode = spark.sql("""
SELECT 
    RatecodeID,
    CASE 
        WHEN RatecodeID = 1 THEN 'Standard rate'
        WHEN RatecodeID = 2 THEN 'JFK'
        WHEN RatecodeID = 3 THEN 'Newark'
        WHEN RatecodeID = 4 THEN 'Nassau/Westchester'
        WHEN RatecodeID = 5 THEN 'Negotiated fare'
        ELSE 'Outro/Desconhecido'
    END as tipo_tarifa,
    COUNT(*) as total_viagens,
    ROUND(AVG(fare_amount), 2) as tarifa_media,
    ROUND(AVG(trip_distance), 2) as distancia_media_mi
FROM taxi_data
GROUP BY RatecodeID
ORDER BY total_viagens DESC
""")
display(ratecode)

## 📋 Resumo — Fase 2: Análise Exploratória

### Qualidade dos Dados
- ✅ Nulos: praticamente inexistentes
- ✅ Duplicatas: apenas 4 em 46,9M (irrelevante)
- ✅ Outliers extremos: <300 registros no total (0,0006%)
- ⚠️ RatecodeID tem 2 códigos não documentados (6 e 99), com volume irrelevante (607 registros)

### Padrões Temporais
- Pico de viagens: 18h-19h (final de tarde)
- Menor volume: 3h-4h da madrugada
- Sábado tem o maior volume, mas a menor tarifa média

### Descoberta: 2015 vs 2016 (Janeiro)
- Medianas de distância IDÊNTICAS (1,70 milhas) — dados consistentes entre anos
- Diferença nas médias era causada só por outliers extremos, não por erro de unidade

### Correlação Distância-Tarifa
- Com outliers: 0.01 (enganoso)
- Sem outliers: **0.95** (forte, como esperado)

### Gorjetas
- Cartão de crédito: 21,3% de gorjeta média
- Dinheiro: 0% (não é erro — gorjetas em dinheiro não são registradas pelo sistema NYC TLC)

### Viagens de Aeroporto (via RatecodeID)
- JFK: 891.844 viagens, tarifa média $52,54 — **bate com a tarifa fixa histórica real de $52** ✅ (ótima validação de qualidade dos dados)
- Newark: 70.036 viagens, tarifa média $66,04
- Nassau/Westchester: distância média (106 mi) parece inflada por outliers — volume pequeno (18k viagens), investigar na Fase 3

### Correção Importante
- `trip_distance` está em **milhas**, não km (confirmado pelo dicionário oficial da TLC)

### Regras de Limpeza Definidas Para a Fase 3
- trip_distance: 0.1 a 200 milhas
- fare_amount: $2.5 a $500
- tip_amount ≥ 0
- total_amount ≥ 0